# Fisher Merging in Task-Vector (Delta) Space for IF + Math

This notebook implements a **delta-space** variant of Fisher Merging (Matena & Raffel, 2022)
for Qwen3-1.7B IF/Math checkpoints.

## Why delta-space?

The original Fisher merging formula operates on **absolute parameter values**:

`theta*_j = sum_i(lambda_i * F_{i,j} * theta_{i,j}) / (sum_i(lambda_i * F_{i,j}) + epsilon)`

This causes a **zero-precision catastrophe** in RL fine-tuning settings: when `F_{i,j} ≈ 0`
for all tasks (12.94% of parameters in our case), the merged parameter collapses to zero
instead of preserving the pretrained value. This destroys the base model's knowledge.

## Delta-space formula (this notebook)

We compute **task vectors** `Delta_{i,j} = theta_{i,j} - theta_base_j` and merge in delta space:

`theta*_j = theta_base_j + sum_i(lambda_i * F_{i,j} * Delta_{i,j}) / (sum_i(lambda_i * F_{i,j}) + epsilon)`

**Key property**: When `F = 0` for all tasks → `theta* = theta_base` (pretrained value preserved).

## Pipeline

1. Load validation parquet and precomputed `correct_rollout_trajectories.parquet` for each task.
2. Subsample a small number of correct rollouts per task (configurable) for fast recomputation.
3. Recompute empirical Fisher diagonal directly from sampled correct responses (no `torchrun`).
4. Apply **delta-space** Fisher precision-weighted merge and save the merged checkpoint.


In [ ]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Mapping, MutableMapping

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class TaskSpec:
    """Task model and dataset specification.

    Args:
        name: Short task key used in output naming.
        model_path: Local Hugging Face checkpoint path for the task model.
        validation_parquet: Source validation parquet containing prompt messages.
        correct_rollout_parquet: Parquet containing only correct rollouts
            (`sample_index`, `output`, etc.) generated in a previous pipeline run.
    """

    name: str
    model_path: Path
    validation_parquet: Path
    correct_rollout_parquet: Path


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for notebook execution.

    Args:
        base_model_id: Base model id/path used for merge architecture/tokenizer.
        output_root: Output root directory for all Fisher and merge artifacts.
        seed: Random seed for reproducibility.
        model_dtype_name: Dtype string for model loading (`bf16`, `fp16`, `fp32`).
        fisher_device: Device used for Fisher gradient computation.
    """

    base_model_id: str
    output_root: Path
    seed: int
    model_dtype_name: str
    fisher_device: str


@dataclass(frozen=True)
class FisherConfig:
    """Configuration for sampled correct-rollout Fisher computation.

    Args:
        max_prompt_tokens: Prompt truncation length before response concatenation.
        max_response_tokens: Response truncation length for gradient computation.
        max_correct_trajectories: Maximum number of correct rollouts sampled per task.
            `None` means use all rows from `correct_rollout_parquet`.
        log_every: Progress logging interval in processed trajectories.
        grad_checkpointing: Whether to enable model gradient checkpointing.
    """

    max_prompt_tokens: int = 1536
    max_response_tokens: int = 512
    max_correct_trajectories: int | None = 128
    log_every: int = 10
    grad_checkpointing: bool = True


@dataclass(frozen=True)
class MergeConfig:
    """Configuration for Fisher precision merge.

    Args:
        epsilon: Numerical stabilizer for denominator.
        if_lambda: Lambda coefficient for IF model.
        math_lambda: Lambda coefficient for Math model.
    """

    epsilon: float = 1e-12
    if_lambda: float = 1.0
    math_lambda: float = 1.0

    def to_task_lambdas(self) -> Dict[str, float]:
        """Convert scalar lambda fields into task-keyed mapping."""

        return {
            "if": float(self.if_lambda),
            "math": float(self.math_lambda),
        }


BASE_MODEL_ID = "Qwen/Qwen3-1.7B"

IF_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface")
MATH_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface")

IF_VALIDATION_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/verl_nemotron_merge/merging_analysis/merging_analysis/artifacts/jwcm_v2/validation_data/val_if_512.parquet"
)
MATH_VALIDATION_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/verl_nemotron_merge/merging_analysis/merging_analysis/artifacts/jwcm_v2/validation_data/val_math_512.parquet"
)

# Reuse previously generated "correct only" rollout tables.
IF_CORRECT_ROLLOUT_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/qwen3_1.7b_fisher_pipeline_verbose/tasks/if/correct_rollout_trajectories.parquet"
)
MATH_CORRECT_ROLLOUT_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/qwen3_1.7b_fisher_pipeline_verbose/tasks/math/correct_rollout_trajectories.parquet"
)

TASK_SPECS: List[TaskSpec] = [
    TaskSpec(
        name="if",
        model_path=IF_MODEL_PATH,
        validation_parquet=IF_VALIDATION_PATH,
        correct_rollout_parquet=IF_CORRECT_ROLLOUT_PATH,
    ),
    TaskSpec(
        name="math",
        model_path=MATH_MODEL_PATH,
        validation_parquet=MATH_VALIDATION_PATH,
        correct_rollout_parquet=MATH_CORRECT_ROLLOUT_PATH,
    ),
]

RUNTIME = RuntimeConfig(
    base_model_id=BASE_MODEL_ID,
    output_root=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge"),
    seed=42,
    model_dtype_name="bf16",
    fisher_device="cuda:0",
)

FISHER_CFG = FisherConfig(
    max_prompt_tokens=1536,
    max_response_tokens=512,
    max_correct_trajectories=128,
    log_every=10,
    grad_checkpointing=True,
)

MERGE_CFG = MergeConfig(
    epsilon=1e-12,
    if_lambda=1.0,
    math_lambda=1.0,
)

for required_path in [
    IF_MODEL_PATH,
    MATH_MODEL_PATH,
    IF_VALIDATION_PATH,
    MATH_VALIDATION_PATH,
    IF_CORRECT_ROLLOUT_PATH,
    MATH_CORRECT_ROLLOUT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required path does not exist: {required_path}")

RUNTIME.output_root.mkdir(parents=True, exist_ok=True)

print(f"Output root: {RUNTIME.output_root}")
print(f"Fisher device: {RUNTIME.fisher_device}")
print(f"Fisher sample cap per task: {FISHER_CFG.max_correct_trajectories}")
for task_spec in TASK_SPECS:
    print(f"Task={task_spec.name} | correct_rollout_parquet={task_spec.correct_rollout_parquet}")


In [ ]:
def set_seed(seed: int) -> None:
    """Set all random seeds for reproducible sampling and gradients.

    Args:
        seed: Integer random seed.

    Returns:
        None. RNG states are updated in-place.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    """Return UTC ISO-8601 timestamp string."""

    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def to_json_compatible(obj: Any) -> Any:
    """Recursively convert runtime objects into JSON-serializable values.

    Args:
        obj: Arbitrary Python object.

    Returns:
        JSON-compatible object composed of dict/list/str/int/float/bool/null.
    """

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, torch.dtype):
        return str(obj)

    if isinstance(obj, dict):
        return {str(key): to_json_compatible(value) for key, value in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [to_json_compatible(value) for value in obj]

    return obj


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Save JSON payload to disk with runtime-object conversion.

    Args:
        payload: JSON mapping that may include Path, tuple, or torch dtype values.
        output_path: Destination file path.

    Returns:
        None. JSON file is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    json_payload = to_json_compatible(dict(payload))
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(json_payload, file, indent=2, ensure_ascii=False)


def resolve_torch_dtype(dtype_name: str) -> torch.dtype:
    """Resolve dtype name into torch dtype object.

    Args:
        dtype_name: Dtype alias string.

    Returns:
        Torch dtype object.
    """

    lookup = {
        "bf16": torch.bfloat16,
        "fp16": torch.float16,
        "fp32": torch.float32,
    }
    if dtype_name not in lookup:
        raise ValueError(f"Unsupported dtype: {dtype_name}")
    return lookup[dtype_name]


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional mistral-regex compatibility argument.

    Args:
        model_name_or_path: HF model id or local path.

    Returns:
        Loaded tokenizer.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def normalize_prompt_messages(prompt_obj: Any) -> List[Dict[str, str]]:
    """Normalize parquet prompt object into chat-template message dictionaries.

    Args:
        prompt_obj: Prompt object loaded from parquet.

    Returns:
        List of dictionaries with `role` and `content` keys.
    """

    if isinstance(prompt_obj, np.ndarray):
        messages = prompt_obj.tolist()
    elif isinstance(prompt_obj, list):
        messages = prompt_obj
    else:
        raise TypeError(f"Unsupported prompt type: {type(prompt_obj)}")

    normalized: List[Dict[str, str]] = []
    for message in messages:
        if not isinstance(message, dict):
            raise TypeError(f"Prompt message must be dict, got {type(message)}")

        role = str(message.get("role", "user"))
        content = str(message.get("content", ""))
        normalized.append({"role": role, "content": content})

    return normalized


def build_index_to_prompt_map(task_spec: TaskSpec) -> Dict[int, List[Dict[str, str]]]:
    """Build source-row-position to prompt-message mapping.

    Why row position is used:
    - `correct_rollout_trajectories.parquet` stores `sample_index` values emitted by
      the reward wrapper from VERL `extra_info["index"]`.
    - In this project pipeline, that index corresponds to source row position, not
      the original dataset's semantic `index` column.

    Args:
        task_spec: Task-level source definition.

    Returns:
        Mapping from `sample_index` to normalized prompt message list.
    """

    source_df = pd.read_parquet(task_spec.validation_parquet)
    if "prompt" not in source_df.columns:
        raise ValueError(
            f"Validation parquet for task '{task_spec.name}' must include 'prompt'. "
            f"columns={list(source_df.columns)}"
        )

    index_to_prompt: Dict[int, List[Dict[str, str]]] = {}
    for row_position, prompt_obj in enumerate(source_df["prompt"].tolist()):
        index_to_prompt[int(row_position)] = normalize_prompt_messages(prompt_obj)
    return index_to_prompt


set_seed(RUNTIME.seed)

fisher_dir = RUNTIME.output_root / "fisher_diagonal"
metadata_dir = RUNTIME.output_root / "metadata"

fisher_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

# Keep the original variable name for downstream merge metadata compatibility.
NORMALIZED_VALIDATION_PATHS: Dict[str, Path] = {
    task_spec.name: task_spec.validation_parquet for task_spec in TASK_SPECS
}
FISHER_OUTPUT_PATHS: Dict[str, Path] = {
    task_spec.name: fisher_dir / f"fisher_diag_{task_spec.name}.pt" for task_spec in TASK_SPECS
}
FISHER_SUMMARY_PATHS: Dict[str, Path] = {
    task_spec.name: fisher_dir / f"fisher_diag_{task_spec.name}_summary.json" for task_spec in TASK_SPECS
}

for task_spec in TASK_SPECS:
    source_row_count = int(pd.read_parquet(task_spec.validation_parquet, columns=["prompt"]).shape[0])
    correct_row_count = int(pd.read_parquet(task_spec.correct_rollout_parquet, columns=["sample_index"]).shape[0])
    print(
        f"Task={task_spec.name} | source_rows={source_row_count} "
        f"| correct_rollout_rows={correct_row_count}"
    )

save_json(
    {
        "created_at": now_iso(),
        "runtime": asdict(RUNTIME),
        "fisher_config": asdict(FISHER_CFG),
        "task_models": {task.name: str(task.model_path) for task in TASK_SPECS},
        "source_validation_parquets": {task.name: str(task.validation_parquet) for task in TASK_SPECS},
        "correct_rollout_parquets": {task.name: str(task.correct_rollout_parquet) for task in TASK_SPECS},
        "fisher_output_paths": {name: str(path) for name, path in FISHER_OUTPUT_PATHS.items()},
        "fisher_summary_paths": {name: str(path) for name, path in FISHER_SUMMARY_PATHS.items()},
    },
    metadata_dir / "fisher_validation_manifest.json",
)

print(f"Saved Fisher manifest: {metadata_dir / 'fisher_validation_manifest.json'}")


In [ ]:
def load_fisher_model_and_tokenizer(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
    device: str,
    grad_checkpointing: bool,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load causal LM and tokenizer for Fisher recomputation.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        torch_dtype: Weight dtype used during model loading.
        device: Runtime device string.
        grad_checkpointing: Whether to enable activation checkpointing.

    Returns:
        Tuple `(model, tokenizer)` ready for gradient computation.
    """

    resolved_path = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved_path,
        torch_dtype=torch_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.to(device)
    model.eval()

    if grad_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        # Activation checkpointing reduces memory spikes during backward.
        model.gradient_checkpointing_enable()
    if hasattr(model, "config") and hasattr(model.config, "use_cache"):
        # `use_cache` must be disabled when checkpointing/backward is active.
        model.config.use_cache = False

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved_path)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def compute_sequence_log_prob_gradients(
    model: AutoModelForCausalLM,
    param_list: List[torch.nn.Parameter],
    full_ids_cpu: torch.Tensor,
    prompt_len: int,
    device: str,
) -> tuple[torch.Tensor | None, ...]:
    """Compute gradients of sequence log-probability over response tokens.

    Args:
        model: Task model in eval mode.
        param_list: Trainable floating parameters tracked for Fisher.
        full_ids_cpu: Concatenated prompt+response token ids on CPU.
        prompt_len: Prompt token count inside `full_ids_cpu`.
        device: Runtime device.

    Returns:
        Tuple of gradients aligned with `param_list`.
    """

    full_ids = full_ids_cpu.unsqueeze(0).to(device)
    full_attention = torch.ones_like(full_ids, device=device)

    logits = model(input_ids=full_ids, attention_mask=full_attention).logits
    full_len = int(full_ids.shape[1])

    shifted_positions = torch.arange(prompt_len, full_len, device=device) - 1
    target_token_ids = full_ids[0, prompt_len:full_len]

    selected_logits = logits[0, shifted_positions, :].to(torch.float32)
    selected_log_probs = torch.log_softmax(selected_logits, dim=-1)
    sequence_log_prob = selected_log_probs.gather(
        dim=1,
        index=target_token_ids.unsqueeze(1),
    ).sum()

    grads = torch.autograd.grad(
        sequence_log_prob,
        param_list,
        retain_graph=False,
        create_graph=False,
        allow_unused=True,
    )

    del logits
    del selected_logits
    del selected_log_probs
    del sequence_log_prob
    return grads


def compute_fisher_from_correct_rollouts(
    task_spec: TaskSpec,
    runtime: RuntimeConfig,
    fisher_cfg: FisherConfig,
    fisher_output_path: Path,
    summary_output_path: Path,
) -> None:
    """Recompute Fisher from sampled correct rollouts for one task.

    Args:
        task_spec: Task model + source/correct-rollout paths.
        runtime: Runtime settings with device/dtype/seed.
        fisher_cfg: Fisher hyperparameters and sampling cap.
        fisher_output_path: Destination `.pt` path for Fisher tensor mapping.
        summary_output_path: Destination `.json` path for run metadata.

    Returns:
        None. Saves Fisher tensors and summary to disk.
    """

    preferred_device = str(runtime.fisher_device)
    if preferred_device.startswith("cuda") and not torch.cuda.is_available():
        # Fallback keeps notebook runnable on CPU-only environments.
        print(
            f"[Warning][{task_spec.name}] CUDA is unavailable; "
            f"falling back from {preferred_device} to cpu."
        )
        worker_device = "cpu"
    else:
        worker_device = preferred_device

    fisher_dtype = resolve_torch_dtype(runtime.model_dtype_name)
    model, tokenizer = load_fisher_model_and_tokenizer(
        model_name_or_path=task_spec.model_path,
        torch_dtype=fisher_dtype,
        device=worker_device,
        grad_checkpointing=fisher_cfg.grad_checkpointing,
    )

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True

    correct_df = pd.read_parquet(task_spec.correct_rollout_parquet)
    if correct_df.empty:
        raise ValueError(f"Task '{task_spec.name}' has empty correct rollout parquet: {task_spec.correct_rollout_parquet}")

    # Deterministic sampling makes repeated notebook runs directly comparable.
    if (
        fisher_cfg.max_correct_trajectories is not None
        and len(correct_df) > int(fisher_cfg.max_correct_trajectories)
    ):
        fisher_df = correct_df.sample(
            n=int(fisher_cfg.max_correct_trajectories),
            random_state=int(runtime.seed),
            replace=False,
        ).reset_index(drop=True)
    else:
        fisher_df = correct_df.reset_index(drop=True)

    index_to_prompt = build_index_to_prompt_map(task_spec=task_spec)
    prompt_token_cache: Dict[int, torch.Tensor] = {}

    param_entries: List[tuple[str, torch.nn.Parameter]] = [
        (name, parameter)
        for name, parameter in model.named_parameters()
        if parameter.requires_grad and torch.is_floating_point(parameter)
    ]
    param_names = [name for name, _ in param_entries]
    param_list = [parameter for _, parameter in param_entries]

    # Keep accumulation on CPU to avoid persistent GPU memory growth.
    fisher_sum_cpu: Dict[str, torch.Tensor] = {
        name: torch.zeros_like(parameter, dtype=torch.float32, device="cpu")
        for name, parameter in param_entries
    }

    processed = 0
    skipped_invalid_index = 0
    skipped_missing_prompt = 0
    skipped_empty_response = 0
    skipped_too_short = 0

    iterator = tqdm(
        fisher_df.to_dict(orient="records"),
        total=len(fisher_df),
        desc=f"Fisher[{task_spec.name}]",
    )
    for step_index, row in enumerate(iterator, start=1):
        sample_index_raw = row.get("sample_index", None)
        try:
            sample_index = int(sample_index_raw)
        except Exception:
            skipped_invalid_index += 1
            continue

        prompt_messages = index_to_prompt.get(sample_index, None)
        if prompt_messages is None:
            skipped_missing_prompt += 1
            continue

        if sample_index not in prompt_token_cache:
            prompt_text = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            prompt_ids_cpu = tokenizer(
                prompt_text,
                return_tensors="pt",
                truncation=True,
                max_length=int(fisher_cfg.max_prompt_tokens),
                add_special_tokens=False,
            )["input_ids"][0].detach().cpu()
            prompt_token_cache[sample_index] = prompt_ids_cpu

        response_text = str(row.get("output", ""))
        response_ids_cpu = tokenizer(
            response_text,
            return_tensors="pt",
            truncation=True,
            max_length=int(fisher_cfg.max_response_tokens),
            add_special_tokens=False,
        )["input_ids"][0].detach().cpu()

        if response_ids_cpu.numel() == 0:
            skipped_empty_response += 1
            continue

        prompt_ids_cpu = prompt_token_cache[sample_index]
        full_ids_cpu = torch.cat([prompt_ids_cpu, response_ids_cpu], dim=0)
        prompt_len = int(prompt_ids_cpu.numel())
        full_len = int(full_ids_cpu.numel())
        if full_len <= prompt_len:
            skipped_too_short += 1
            continue

        try:
            grads = compute_sequence_log_prob_gradients(
                model=model,
                param_list=param_list,
                full_ids_cpu=full_ids_cpu,
                prompt_len=prompt_len,
                device=worker_device,
            )
            for param_index, grad in enumerate(grads):
                if grad is None:
                    continue
                name = param_names[param_index]
                fisher_sum_cpu[name].add_(grad.detach().to(torch.float32).cpu().pow(2))
            processed += 1
            del grads
        except torch.OutOfMemoryError as oom_error:
            raise RuntimeError(
                f"[{task_spec.name}] Fisher OOM at sample_index={sample_index}. "
                f"Reduce FisherConfig.max_response_tokens/max_prompt_tokens "
                f"or use a smaller max_correct_trajectories."
            ) from oom_error
        finally:
            model.zero_grad(set_to_none=True)

        if step_index % int(fisher_cfg.log_every) == 0:
            print(
                f"[{task_spec.name}] progress processed={processed}/{len(fisher_df)} "
                f"(step={step_index})"
            )

        if processed > 0 and processed % 16 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if processed <= 0:
        raise RuntimeError(
            f"Task '{task_spec.name}' produced zero valid trajectories after filtering. "
            f"Check sample_index alignment and response tokenization."
        )

    for name in fisher_sum_cpu.keys():
        fisher_sum_cpu[name].div_(float(processed))

    fisher_output_path.parent.mkdir(parents=True, exist_ok=True)
    summary_output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(fisher_sum_cpu, fisher_output_path)

    summary = {
        "created_at": now_iso(),
        "task_name": task_spec.name,
        "task_model_path": str(task_spec.model_path),
        "validation_parquet": str(task_spec.validation_parquet),
        "correct_rollout_parquet": str(task_spec.correct_rollout_parquet),
        "fisher_output_path": str(fisher_output_path),
        "dtype": runtime.model_dtype_name,
        "device": worker_device,
        "max_prompt_tokens": int(fisher_cfg.max_prompt_tokens),
        "max_response_tokens": int(fisher_cfg.max_response_tokens),
        "max_correct_trajectories": None
        if fisher_cfg.max_correct_trajectories is None
        else int(fisher_cfg.max_correct_trajectories),
        "num_correct_rollout_rows": int(len(correct_df)),
        "num_sampled_rollout_rows": int(len(fisher_df)),
        "processed_sequences": int(processed),
        "skipped_invalid_index": int(skipped_invalid_index),
        "skipped_missing_prompt": int(skipped_missing_prompt),
        "skipped_empty_response": int(skipped_empty_response),
        "skipped_too_short": int(skipped_too_short),
        "num_tracked_parameters": int(len(param_entries)),
    }
    save_json(summary, summary_output_path)

    print(f"[{task_spec.name}] Saved Fisher diagonal: {fisher_output_path}")
    print(f"[{task_spec.name}] Saved Fisher summary: {summary_output_path}")

    del model
    del tokenizer
    del fisher_sum_cpu
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


for task_spec in TASK_SPECS:
    compute_fisher_from_correct_rollouts(
        task_spec=task_spec,
        runtime=RUNTIME,
        fisher_cfg=FISHER_CFG,
        fisher_output_path=FISHER_OUTPUT_PATHS[task_spec.name],
        summary_output_path=FISHER_SUMMARY_PATHS[task_spec.name],
    )

print("Sampled correct-rollout Fisher recomputation finished for all tasks.")


In [ ]:
def load_causal_lm(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load causal LM and tokenizer on target device.

    Args:
        model_name_or_path: HF model id or local checkpoint path.
        torch_dtype: Weight dtype for model loading.
        device: Target device (`cpu` or `cuda`).

    Returns:
        Tuple `(model, tokenizer)`.
    """

    resolved = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved,
        torch_dtype=torch_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.to(device)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def validate_parameter_compatibility(models: Mapping[str, AutoModelForCausalLM]) -> None:
    """Validate parameter-key and shape compatibility among models.

    Args:
        models: Mapping from model-name key to loaded model.

    Returns:
        None. Raises `ValueError` on incompatibility.
    """

    items = list(models.items())
    if not items:
        raise ValueError("No models were provided for compatibility check.")

    ref_name, ref_model = items[0]
    ref_params = dict(ref_model.named_parameters())

    for current_name, current_model in items[1:]:
        current_params = dict(current_model.named_parameters())

        if set(ref_params.keys()) != set(current_params.keys()):
            missing_in_current = sorted(set(ref_params.keys()) - set(current_params.keys()))
            missing_in_ref = sorted(set(current_params.keys()) - set(ref_params.keys()))
            raise ValueError(
                f"Parameter keys mismatch between {ref_name} and {current_name}. "
                f"missing_in_{current_name}={missing_in_current[:5]}, "
                f"missing_in_{ref_name}={missing_in_ref[:5]}"
            )

        for param_name in ref_params.keys():
            if ref_params[param_name].shape != current_params[param_name].shape:
                raise ValueError(
                    f"Shape mismatch for '{param_name}' between {ref_name} and {current_name}: "
                    f"{tuple(ref_params[param_name].shape)} vs {tuple(current_params[param_name].shape)}"
                )


def validate_fisher_compatibility(
    task_models: Mapping[str, AutoModelForCausalLM],
    fisher_by_task: Mapping[str, Mapping[str, torch.Tensor]],
) -> None:
    """Validate Fisher tensor availability and shapes for all task parameters.

    Args:
        task_models: Loaded task model mapping.
        fisher_by_task: Fisher tensors keyed by task and parameter name.

    Returns:
        None. Raises `ValueError` when a mismatch is found.
    """

    for task_name, model in task_models.items():
        if task_name not in fisher_by_task:
            raise ValueError(f"Missing fisher tensors for task '{task_name}'")

        fisher_params = fisher_by_task[task_name]
        for param_name, param_tensor in model.named_parameters():
            if not torch.is_floating_point(param_tensor.data):
                continue
            if param_name not in fisher_params:
                raise ValueError(
                    f"Missing fisher tensor for task '{task_name}' and parameter '{param_name}'"
                )
            if tuple(fisher_params[param_name].shape) != tuple(param_tensor.shape):
                raise ValueError(
                    f"Fisher shape mismatch for task '{task_name}', parameter '{param_name}': "
                    f"fisher={tuple(fisher_params[param_name].shape)} "
                    f"model={tuple(param_tensor.shape)}"
                )


def merge_with_fisher_precision_inplace(
    merge_model: AutoModelForCausalLM,
    task_models: Mapping[str, AutoModelForCausalLM],
    fisher_by_task: Mapping[str, Mapping[str, torch.Tensor]],
    task_lambdas: Mapping[str, float],
    epsilon: float,
) -> Dict[str, Any]:
    """Apply Fisher precision-weighted merge in task-vector (delta) space in-place.

    Why this refactor is necessary:
        The original absolute-parameter Fisher merge can destroy pretrained weights
        when all task Fishers are zero for a parameter. In that case,
        numerator ~= 0 and denominator ~= 0, so the merged value can collapse
        toward zero instead of preserving the base model.

    Delta-space formula used here:
        Delta_{i,j} = theta_{i,j} - theta_base_j
        theta*_j = theta_base_j
                   + sum_i(lambda_i * F_{i,j} * Delta_{i,j})
                   / (sum_i(lambda_i * F_{i,j}) + epsilon)

    Safety behavior:
        For denominator <= epsilon, this implementation explicitly falls back to
        theta_base_j to guarantee pretrained anchor preservation.

    Args:
        merge_model: Base model whose parameters serve as theta_base.
                     Overwritten in-place with merged parameters.
        task_models: Source task models (fine-tuned from the same base).
        fisher_by_task: Per-task Fisher diagonal dictionaries.
        task_lambdas: Task-level lambda coefficients.
        epsilon: Numerical stabilizer for denominator and zero-precision threshold.

    Returns:
        Summary dictionary including zero-precision diagnostics and fallback stats.
    """

    # Include base model in compatibility check to guarantee aligned parameter keys/shapes.
    compatibility_models: Dict[str, AutoModelForCausalLM] = {"base": merge_model}
    compatibility_models.update({task_name: model for task_name, model in task_models.items()})

    validate_parameter_compatibility(compatibility_models)
    validate_fisher_compatibility(task_models=task_models, fisher_by_task=fisher_by_task)

    merge_named = dict(merge_model.named_parameters())
    task_named = {task_name: dict(model.named_parameters()) for task_name, model in task_models.items()}
    task_names = list(task_models.keys())

    zero_precision_elements = 0
    exact_zero_precision_elements = 0
    total_elements = 0
    max_abs_delta_on_zero_precision_before_fallback = 0.0

    with torch.no_grad():
        for param_name, merge_param in merge_named.items():
            if not torch.is_floating_point(merge_param.data):
                continue

            # theta_base: current value of merge_model (loaded from base checkpoint).
            theta_base = merge_param.data.detach().to(torch.float32)

            numerator = torch.zeros_like(theta_base, dtype=torch.float32)
            denominator = torch.zeros_like(theta_base, dtype=torch.float32)

            for task_name in task_names:
                lambda_i = float(task_lambdas[task_name])
                theta_i = task_named[task_name][param_name].data.detach().to(torch.float32)
                fisher_i = fisher_by_task[task_name][param_name].detach().to(torch.float32)

                # Task-vector (delta) merge: accumulate precision-weighted deltas.
                delta_i = theta_i - theta_base
                weighted_precision = lambda_i * fisher_i

                numerator.add_(weighted_precision * delta_i)
                denominator.add_(weighted_precision)

            # Identify low-precision coordinates where Fisher provides no reliable signal.
            zero_precision_mask = denominator <= float(epsilon)
            exact_zero_mask = denominator == 0.0

            # Compute delta update with stabilized denominator first.
            merged_delta = numerator / (denominator + float(epsilon))

            # Diagnostic: estimate magnitude that would have been applied at low-precision
            # coordinates before explicit base fallback.
            if zero_precision_mask.any():
                max_candidate = float(merged_delta[zero_precision_mask].abs().max().item())
                max_abs_delta_on_zero_precision_before_fallback = max(
                    max_abs_delta_on_zero_precision_before_fallback,
                    max_candidate,
                )

            merged_tensor = theta_base + merged_delta

            # Critical safeguard: preserve pretrained base value where Fisher precision
            # is effectively zero, preventing zero-precision collapse.
            if zero_precision_mask.any():
                merged_tensor = torch.where(zero_precision_mask, theta_base, merged_tensor)

            merge_param.data.copy_(merged_tensor.to(merge_param.dtype))

            zero_precision_elements += int(zero_precision_mask.sum().item())
            exact_zero_precision_elements += int(exact_zero_mask.sum().item())
            total_elements += int(denominator.numel())

    return {
        "epsilon": float(epsilon),
        "task_lambdas": {k: float(v) for k, v in task_lambdas.items()},
        "zero_precision_elements": int(zero_precision_elements),
        "exact_zero_precision_elements": int(exact_zero_precision_elements),
        "total_elements": int(total_elements),
        "zero_precision_ratio": None
        if total_elements == 0
        else float(zero_precision_elements / total_elements),
        "exact_zero_precision_ratio": None
        if total_elements == 0
        else float(exact_zero_precision_elements / total_elements),
        "max_abs_delta_on_zero_precision_before_fallback": float(
            max_abs_delta_on_zero_precision_before_fallback
        ),
        "merge_space": "task_vector_delta_space",
    }



def save_merged_artifacts(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save merged model, tokenizer, and metadata payload.

    Args:
        model: Merged model.
        tokenizer: Tokenizer to save with model.
        output_dir: Destination directory.
        metadata: JSON metadata mapping.

    Returns:
        None.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(dict(metadata), output_dir / "merge_metadata.json")


merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)
fisher_by_task = {
    task_spec.name: torch.load(FISHER_OUTPUT_PATHS[task_spec.name], map_location="cpu")
    for task_spec in TASK_SPECS
}

merge_model, merge_tokenizer = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=merge_dtype,
    device="cpu",
)

task_models: MutableMapping[str, AutoModelForCausalLM] = {}
for task_spec in TASK_SPECS:
    task_model, _ = load_causal_lm(
        model_name_or_path=task_spec.model_path,
        torch_dtype=merge_dtype,
        device="cpu",
    )
    task_models[task_spec.name] = task_model

merge_summary = merge_with_fisher_precision_inplace(
    merge_model=merge_model,
    task_models=task_models,
    fisher_by_task=fisher_by_task,
    task_lambdas=MERGE_CFG.to_task_lambdas(),
    epsilon=MERGE_CFG.epsilon,
)

merged_output_dir = RUNTIME.output_root / "fisher_merge_delta_if_math"
save_merged_artifacts(
    model=merge_model,
    tokenizer=merge_tokenizer,
    output_dir=merged_output_dir,
    metadata={
        "created_at": now_iso(),
        "method": "fisher_precision_weighted_merge_delta_space",
        "formula": "theta*=theta_base+sum_i(lambda_i*F_i*Delta_i)/(sum_i(lambda_i*F_i)+epsilon)",
        "formula_note": "Delta_i = theta_i - theta_base. When F=0, theta* = theta_base (pretrained preserved).",
        "runtime": asdict(RUNTIME),
        "fisher_config": asdict(FISHER_CFG),
        "merge_config": asdict(MERGE_CFG),
        "task_models": {task.name: str(task.model_path) for task in TASK_SPECS},
        "fisher_output_paths": {name: str(path) for name, path in FISHER_OUTPUT_PATHS.items()},
        "fisher_summary_paths": {name: str(path) for name, path in FISHER_SUMMARY_PATHS.items()},
        "normalized_validation_paths": {name: str(path) for name, path in NORMALIZED_VALIDATION_PATHS.items()},
        "merge_summary": merge_summary,
    },
)

run_summary_path = RUNTIME.output_root / "metadata" / "fisher_merge_delta_run_summary.json"
save_json(
    {
        "created_at": now_iso(),
        "runtime": asdict(RUNTIME),
        "fisher_config": asdict(FISHER_CFG),
        "merge_config": asdict(MERGE_CFG),
        "fisher_outputs": {name: str(path) for name, path in FISHER_OUTPUT_PATHS.items()},
        "merged_output_dir": str(merged_output_dir),
        "merge_summary": merge_summary,
    },
    run_summary_path,
)

print(f"Saved merged checkpoint: {merged_output_dir}")
print(f"Saved run summary: {run_summary_path}")

del merge_model
del merge_tokenizer
for task_name in list(task_models.keys()):
    del task_models[task_name]
del fisher_by_task
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Fisher diagonal concentration diagnostics
# This cell helps answer whether Fisher scale is concentrated in a small subset of
# parameters (sparse) or spread relatively uniformly across the whole model.

from pathlib import Path
from typing import Any, Dict, Mapping, Tuple

import matplotlib.pyplot as plt


def load_fisher_dictionary(task_name: str, fisher_path: Path) -> Mapping[str, torch.Tensor]:
    """Load one task's Fisher diagonal dictionary from disk.

    Args:
        task_name: Short task key used for readable error messages.
        fisher_path: Path to the serialized Fisher tensor dictionary (`.pt`).

    Returns:
        Parameter-name keyed Fisher diagonal dictionary on CPU.

    Raises:
        FileNotFoundError: If the expected Fisher file is missing.
        TypeError: If the deserialized object is not a mapping.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f"Fisher file for task '{task_name}' was not found: {fisher_path}")

    fisher_obj = torch.load(fisher_path, map_location='cpu')
    if not isinstance(fisher_obj, Mapping):
        raise TypeError(
            f"Fisher file for task '{task_name}' must contain a mapping, got {type(fisher_obj)}"
        )

    return fisher_obj


def collect_tensor_mass_stats(
    fisher_tensors: Mapping[str, torch.Tensor],
) -> Tuple[pd.DataFrame, int, float]:
    """Collect per-parameter Fisher mass statistics.

    Why this exists:
        Tensor-level mass share reveals whether only a few named parameters/layers
        dominate the total Fisher mass.

    Args:
        fisher_tensors: Mapping from parameter name to Fisher diagonal tensor.

    Returns:
        Tuple of:
        - DataFrame with per-parameter stats (`numel`, `mass`, `mean`, `max`, `mass_share`)
        - Total element count across all tensors
        - Total Fisher mass across all tensors
    """

    rows = []
    total_numel = 0
    total_mass = 0.0

    for parameter_name, fisher_tensor in fisher_tensors.items():
        # We use absolute value defensively in case tiny negative values appear from
        # numerical noise; Fisher diagonals should be non-negative in theory.
        flat_values = fisher_tensor.detach().to(torch.float32).abs().reshape(-1)

        numel = int(flat_values.numel())
        mass = float(flat_values.sum().item())
        mean_value = float(mass / max(numel, 1))
        max_value = float(flat_values.max().item()) if numel > 0 else 0.0

        rows.append(
            {
                'parameter_name': parameter_name,
                'numel': numel,
                'mass': mass,
                'mean': mean_value,
                'max': max_value,
            }
        )

        total_numel += numel
        total_mass += mass

    tensor_stats_df = pd.DataFrame(rows)
    if total_mass > 0.0 and not tensor_stats_df.empty:
        tensor_stats_df['mass_share'] = tensor_stats_df['mass'] / total_mass
    else:
        tensor_stats_df['mass_share'] = 0.0

    return tensor_stats_df, total_numel, total_mass


def sample_fisher_values(
    fisher_tensors: Mapping[str, torch.Tensor],
    total_numel: int,
    sample_size: int,
    seed: int,
) -> np.ndarray:
    """Uniformly sample Fisher elements without materializing one giant vector.

    Why this exists:
        Full flattening across all model parameters is memory-heavy for large LMs.
        This two-pass index strategy gives an unbiased global sample with low memory.

    Args:
        fisher_tensors: Mapping from parameter name to Fisher tensor.
        total_numel: Total number of scalar elements across all tensors.
        sample_size: Number of scalar Fisher values to sample.
        seed: RNG seed for reproducibility.

    Returns:
        1D NumPy array of sampled non-negative Fisher values.
    """

    if total_numel <= 0 or sample_size <= 0:
        return np.zeros(0, dtype=np.float32)

    effective_sample_size = int(min(sample_size, total_numel))
    rng = np.random.default_rng(seed=seed)

    # Sampling with replacement keeps memory predictable even when total_numel is huge.
    sampled_global_indices = np.sort(
        rng.integers(low=0, high=total_numel, size=effective_sample_size, dtype=np.int64)
    )

    sampled_values = np.empty(effective_sample_size, dtype=np.float32)
    write_cursor = 0
    tensor_offset = 0

    for fisher_tensor in fisher_tensors.values():
        flat_values = fisher_tensor.detach().to(torch.float32).abs().reshape(-1)
        tensor_numel = int(flat_values.numel())

        # Locate sampled global indices that fall into the current tensor range.
        left = np.searchsorted(sampled_global_indices, tensor_offset, side='left')
        right = np.searchsorted(sampled_global_indices, tensor_offset + tensor_numel, side='left')

        if right > left:
            local_indices = sampled_global_indices[left:right] - tensor_offset
            local_index_tensor = torch.from_numpy(local_indices.astype(np.int64))
            local_values = flat_values.index_select(dim=0, index=local_index_tensor)

            next_cursor = write_cursor + (right - left)
            sampled_values[write_cursor:next_cursor] = local_values.cpu().numpy()
            write_cursor = next_cursor

        tensor_offset += tensor_numel

    if write_cursor != effective_sample_size:
        raise RuntimeError(
            f"Sampling bookkeeping mismatch: expected {effective_sample_size}, got {write_cursor}"
        )

    return sampled_values


def compute_lorenz_curve(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Lorenz curve points for non-negative values.

    Args:
        values: 1D array of Fisher values.

    Returns:
        Tuple of arrays (`population_share`, `mass_share`).
    """

    if values.size == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])

    clipped = np.clip(values.astype(np.float64), a_min=0.0, a_max=None)
    sorted_values = np.sort(clipped)
    cumulative_mass = np.cumsum(sorted_values)

    total_mass = float(cumulative_mass[-1])
    if total_mass <= 0.0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])

    mass_share = np.concatenate(([0.0], cumulative_mass / total_mass))
    population_share = np.linspace(0.0, 1.0, mass_share.size)
    return population_share, mass_share


def compute_gini_from_lorenz(population_share: np.ndarray, mass_share: np.ndarray) -> float:
    """Compute Gini coefficient from Lorenz curve points.

    Args:
        population_share: X-axis values of Lorenz curve.
        mass_share: Y-axis values of Lorenz curve.

    Returns:
        Gini coefficient in [0, 1], where larger means more concentration.
    """

    area_under_curve = float(np.trapz(mass_share, population_share))
    gini = 1.0 - 2.0 * area_under_curve
    return float(np.clip(gini, 0.0, 1.0))


def compute_top_mass_share(values: np.ndarray, top_fraction: float) -> float:
    """Compute the mass captured by the top fraction of sampled values.

    Args:
        values: Sampled Fisher values.
        top_fraction: Fraction in (0, 1], e.g. 0.01 for top 1%.

    Returns:
        Fraction of total sampled mass carried by top-ranked elements.
    """

    if values.size == 0:
        return 0.0

    clipped = np.clip(values.astype(np.float64), a_min=0.0, a_max=None)
    total_mass = float(clipped.sum())
    if total_mass <= 0.0:
        return 0.0

    top_count = int(max(1, np.ceil(clipped.size * float(top_fraction))))
    top_values = np.partition(clipped, clipped.size - top_count)[-top_count:]
    return float(top_values.sum() / total_mass)


SAMPLE_SIZE_PER_TASK = 500_000
TOP_FRACTIONS = (0.001, 0.01, 0.05, 0.10)
EPSILON_FOR_LOG = 1e-30


task_diagnostics: Dict[str, Dict[str, Any]] = {}

for task_spec in TASK_SPECS:
    task_name = task_spec.name
    fisher_path = Path(FISHER_OUTPUT_PATHS[task_name])

    fisher_tensors = load_fisher_dictionary(task_name=task_name, fisher_path=fisher_path)
    tensor_stats_df, total_numel, total_mass = collect_tensor_mass_stats(fisher_tensors=fisher_tensors)
    sampled_values = sample_fisher_values(
        fisher_tensors=fisher_tensors,
        total_numel=total_numel,
        sample_size=SAMPLE_SIZE_PER_TASK,
        seed=RUNTIME.seed,
    )

    population_share, mass_share = compute_lorenz_curve(sampled_values)

    task_diagnostics[task_name] = {
        'tensor_stats_df': tensor_stats_df,
        'total_numel': total_numel,
        'total_mass': total_mass,
        'sampled_values': sampled_values,
        'population_share': population_share,
        'mass_share': mass_share,
    }

# Build a compact numeric summary so sparsity/uniformity can be compared directly.
summary_rows = []
for task_name, diag in task_diagnostics.items():
    sampled_values = diag['sampled_values']
    tensor_stats_df = diag['tensor_stats_df']

    top_tensor_share = (
        float(tensor_stats_df['mass_share'].max()) if not tensor_stats_df.empty else 0.0
    )
    top10_tensor_share = (
        float(tensor_stats_df['mass_share'].nlargest(min(10, len(tensor_stats_df))).sum())
        if not tensor_stats_df.empty
        else 0.0
    )

    summary_row = {
        'task': task_name,
        'total_elements': int(diag['total_numel']),
        'total_fisher_mass': float(diag['total_mass']),
        'sample_size': int(sampled_values.size),
        'sample_fraction': float(sampled_values.size / max(diag['total_numel'], 1)),
        'gini_sampled': compute_gini_from_lorenz(
            population_share=diag['population_share'], mass_share=diag['mass_share']
        ),
        'top_tensor_mass_share': top_tensor_share,
        'top10_tensor_mass_share': top10_tensor_share,
    }

    for fraction in TOP_FRACTIONS:
        key = f"top_{fraction * 100:.1f}pct_mass_share".replace('.', '_')
        summary_row[key] = compute_top_mass_share(sampled_values, top_fraction=fraction)

    summary_rows.append(summary_row)

summary_df = pd.DataFrame(summary_rows).sort_values('task').reset_index(drop=True)
print('Interpretation hint: higher Gini / higher top-x% mass share means stronger concentration (sparser Fisher scale).')
display(summary_df)

# 2x2 panel to compare element-level and tensor-level concentration patterns.
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
ax_hist, ax_lorenz, ax_topk, ax_tensor_cum = axes.flatten()

for task_name, diag in task_diagnostics.items():
    sampled_values = diag['sampled_values']
    log_values = np.log10(np.clip(sampled_values, a_min=EPSILON_FOR_LOG, a_max=None))

    ax_hist.hist(
        log_values,
        bins=120,
        density=True,
        alpha=0.45,
        label=task_name,
    )

ax_hist.set_title('Sampled Fisher distribution (log10 scale)')
ax_hist.set_xlabel('log10(Fisher + epsilon)')
ax_hist.set_ylabel('Density')
ax_hist.legend()

for task_name, diag in task_diagnostics.items():
    ax_lorenz.plot(diag['population_share'], diag['mass_share'], linewidth=2.0, label=task_name)

ax_lorenz.plot([0.0, 1.0], [0.0, 1.0], 'k--', linewidth=1.0, label='Uniform baseline')
ax_lorenz.set_title('Lorenz curve of sampled Fisher mass')
ax_lorenz.set_xlabel('Fraction of parameters (sampled)')
ax_lorenz.set_ylabel('Cumulative Fisher mass fraction')
ax_lorenz.legend()

fraction_labels = [f"top {fraction * 100:.1f}%" for fraction in TOP_FRACTIONS]
bar_positions = np.arange(len(fraction_labels), dtype=np.float64)
task_names = list(task_diagnostics.keys())
bar_width = 0.8 / max(len(task_names), 1)

for task_index, task_name in enumerate(task_names):
    sampled_values = task_diagnostics[task_name]['sampled_values']
    heights = [compute_top_mass_share(sampled_values, top_fraction=fraction) for fraction in TOP_FRACTIONS]

    offset = (task_index - (len(task_names) - 1) / 2.0) * bar_width
    ax_topk.bar(bar_positions + offset, heights, width=bar_width, label=task_name)

ax_topk.set_xticks(bar_positions)
ax_topk.set_xticklabels(fraction_labels, rotation=0)
ax_topk.set_ylim(0.0, 1.0)
ax_topk.set_title('Mass captured by top-ranked sampled elements')
ax_topk.set_ylabel('Fisher mass fraction')
ax_topk.legend()

for task_name, diag in task_diagnostics.items():
    tensor_stats_df = diag['tensor_stats_df'].sort_values('mass_share', ascending=False)
    if tensor_stats_df.empty:
        continue

    cumulative_mass = tensor_stats_df['mass_share'].cumsum().to_numpy()
    tensor_fraction = np.arange(1, cumulative_mass.size + 1, dtype=np.float64) / cumulative_mass.size
    ax_tensor_cum.plot(tensor_fraction, cumulative_mass, linewidth=2.0, label=task_name)

ax_tensor_cum.plot([0.0, 1.0], [0.0, 1.0], 'k--', linewidth=1.0, label='Uniform baseline')
ax_tensor_cum.set_title('Cumulative mass over parameter tensors')
ax_tensor_cum.set_xlabel('Fraction of parameter tensors (sorted by mass)')
ax_tensor_cum.set_ylabel('Cumulative Fisher mass fraction')
ax_tensor_cum.legend()

plt.tight_layout()
plt.show()

# Print top tensor contributors for quick qualitative inspection.
for task_name, diag in task_diagnostics.items():
    print(f"\nTop 15 parameter tensors by Fisher mass share - task={task_name}")
    display(
        diag['tensor_stats_df']
        .sort_values('mass_share', ascending=False)
        .head(15)
        [['parameter_name', 'numel', 'mass_share', 'mean', 'max']]
        .reset_index(drop=True)
    )


In [ ]:
# Fisher dominant layer/head analysis
# This cell localizes which layers, module groups, and attention heads dominate
# Fisher mass when global concentration metrics (e.g., high Gini) are extreme.

import re
from pathlib import Path
from typing import Dict, Mapping, Optional, Tuple

import matplotlib.pyplot as plt
from transformers import AutoConfig


LAYER_PATTERN = re.compile(r"^model\.layers\.(\d+)\.")
ATTN_QKV_WEIGHT_PATTERN = re.compile(
    r"^model\.layers\.(\d+)\.self_attn\.(q_proj|k_proj|v_proj)\.weight$"
)


def load_fisher_dictionary_for_task(task_name: str, fisher_path: Path) -> Mapping[str, torch.Tensor]:
    """Load one task Fisher dictionary from disk.

    Args:
        task_name: Task key used for diagnostics.
        fisher_path: Path to serialized Fisher-diagonal `.pt` mapping.

    Returns:
        Mapping from parameter name to Fisher tensor.

    Raises:
        FileNotFoundError: If Fisher artifact is missing.
        TypeError: If loaded object is not a mapping.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f"Missing Fisher file for task '{task_name}': {fisher_path}")

    fisher_obj = torch.load(fisher_path, map_location='cpu')
    if not isinstance(fisher_obj, Mapping):
        raise TypeError(
            f"Fisher object for task '{task_name}' must be a mapping, got {type(fisher_obj)}"
        )

    return fisher_obj


def extract_layer_index(parameter_name: str) -> Optional[int]:
    """Extract transformer layer index from parameter name.

    Args:
        parameter_name: Named parameter key in Hugging Face model format.

    Returns:
        Integer layer index when key belongs to `model.layers.{idx}.*`, else `None`.
    """

    match = LAYER_PATTERN.match(parameter_name)
    if match is None:
        return None
    return int(match.group(1))


def classify_module_group(parameter_name: str) -> str:
    """Classify parameter into interpretable module groups.

    Why this exists:
        Group-level aggregation (attn/MLP/norm/embedding) makes it easy to see
        whether Fisher concentration is dominated by a specific subsystem.

    Args:
        parameter_name: Named parameter key.

    Returns:
        Module-group label string.
    """

    if parameter_name.startswith('model.embed_tokens.'):
        return 'embed_tokens'
    if parameter_name.startswith('lm_head.'):
        return 'lm_head'
    if parameter_name.startswith('model.norm.'):
        return 'final_norm'

    if '.self_attn.q_proj.' in parameter_name:
        return 'attn_q_proj'
    if '.self_attn.k_proj.' in parameter_name:
        return 'attn_k_proj'
    if '.self_attn.v_proj.' in parameter_name:
        return 'attn_v_proj'
    if '.self_attn.o_proj.' in parameter_name:
        return 'attn_o_proj'

    if '.mlp.gate_proj.' in parameter_name:
        return 'mlp_gate_proj'
    if '.mlp.up_proj.' in parameter_name:
        return 'mlp_up_proj'
    if '.mlp.down_proj.' in parameter_name:
        return 'mlp_down_proj'

    if '.input_layernorm.' in parameter_name:
        return 'input_layernorm'
    if '.post_attention_layernorm.' in parameter_name:
        return 'post_attention_layernorm'

    return 'other'


def format_layer_label(layer_index: int) -> str:
    """Format integer layer index into compact display label."""

    if layer_index < 0:
        return 'non_transformer'
    return f"L{layer_index:02d}"


def collect_parameter_mass_dataframe(
    fisher_tensors: Mapping[str, torch.Tensor],
) -> Tuple[pd.DataFrame, float]:
    """Build per-parameter Fisher mass table.

    Args:
        fisher_tensors: Fisher diagonal tensors keyed by parameter name.

    Returns:
        Tuple of:
        - DataFrame with per-parameter metadata and Fisher mass statistics.
        - Total Fisher mass across all parameters.
    """

    rows = []
    total_mass = 0.0

    for parameter_name, fisher_tensor in fisher_tensors.items():
        # Absolute value is used defensively against tiny numerical negatives.
        fisher_abs = fisher_tensor.detach().to(torch.float32).abs()

        parameter_mass = float(fisher_abs.sum().item())
        parameter_numel = int(fisher_abs.numel())

        layer_index = extract_layer_index(parameter_name)
        normalized_layer_index = int(layer_index) if layer_index is not None else -1

        rows.append(
            {
                'parameter_name': parameter_name,
                'layer_index': normalized_layer_index,
                'layer_label': format_layer_label(normalized_layer_index),
                'module_group': classify_module_group(parameter_name),
                'numel': parameter_numel,
                'mass': parameter_mass,
                'mean': float(parameter_mass / max(parameter_numel, 1)),
                'max': float(fisher_abs.max().item()) if parameter_numel > 0 else 0.0,
            }
        )

        total_mass += parameter_mass

    parameter_df = pd.DataFrame(rows)
    if parameter_df.empty:
        return parameter_df, total_mass

    if total_mass > 0.0:
        parameter_df['mass_share_global'] = parameter_df['mass'] / total_mass
    else:
        parameter_df['mass_share_global'] = 0.0

    return parameter_df, total_mass


def build_layer_module_summaries(parameter_df: pd.DataFrame, total_mass: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Aggregate parameter Fisher mass to layer and layer-module levels.

    Args:
        parameter_df: Per-parameter Fisher mass table.
        total_mass: Total Fisher mass across all parameters.

    Returns:
        Tuple of:
        - Layer-level summary table.
        - Layer+module-group summary table.
    """

    if parameter_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    layer_df = (
        parameter_df
        .groupby(['layer_index', 'layer_label'], as_index=False)
        .agg(mass=('mass', 'sum'), numel=('numel', 'sum'))
    )
    layer_df['mass_share_global'] = 0.0 if total_mass <= 0.0 else (layer_df['mass'] / total_mass)
    layer_df['mean'] = layer_df['mass'] / layer_df['numel'].clip(lower=1)
    layer_df = layer_df.sort_values('mass_share_global', ascending=False).reset_index(drop=True)

    layer_module_df = (
        parameter_df
        .groupby(['layer_index', 'layer_label', 'module_group'], as_index=False)
        .agg(mass=('mass', 'sum'), numel=('numel', 'sum'))
    )
    layer_module_df['mass_share_global'] = (
        0.0 if total_mass <= 0.0 else (layer_module_df['mass'] / total_mass)
    )
    layer_module_df['mean'] = layer_module_df['mass'] / layer_module_df['numel'].clip(lower=1)
    layer_module_df = layer_module_df.sort_values('mass_share_global', ascending=False).reset_index(drop=True)

    return layer_df, layer_module_df


def resolve_num_attention_heads(reference_model_path: Path) -> Optional[int]:
    """Read number of attention heads from model config.

    Args:
        reference_model_path: Local model path with HF config.

    Returns:
        `num_attention_heads` when available, otherwise `None`.
    """

    try:
        config = AutoConfig.from_pretrained(str(reference_model_path), trust_remote_code=True)
    except Exception as exc:
        print(f"[warn] Failed to load AutoConfig from {reference_model_path}: {exc}")
        return None

    num_heads = getattr(config, 'num_attention_heads', None)
    if num_heads is None:
        print('[warn] num_attention_heads is missing in model config; head analysis is skipped.')
        return None

    return int(num_heads)


def collect_attention_head_mass(
    fisher_tensors: Mapping[str, torch.Tensor],
    num_attention_heads: int,
    total_mass: float,
) -> pd.DataFrame:
    """Compute per-layer per-head Fisher mass from Q/K/V projection weights.

    Important note:
        This decomposition assumes Q/K/V projection output channels are grouped
        contiguously by head, which matches standard transformer implementations.

    Args:
        fisher_tensors: Fisher diagonal tensors keyed by parameter name.
        num_attention_heads: Number of attention heads from model config.
        total_mass: Total Fisher mass across all parameters.

    Returns:
        DataFrame with per-head Fisher mass and global mass share.
    """

    rows = []

    for parameter_name, fisher_tensor in fisher_tensors.items():
        match = ATTN_QKV_WEIGHT_PATTERN.match(parameter_name)
        if match is None:
            continue

        layer_index = int(match.group(1))
        projection_name = str(match.group(2))

        fisher_abs = fisher_tensor.detach().to(torch.float32).abs()
        if fisher_abs.ndim != 2:
            # Q/K/V projection weights are expected to be 2D linear matrices.
            continue

        out_features = int(fisher_abs.shape[0])
        in_features = int(fisher_abs.shape[1])
        if out_features % int(num_attention_heads) != 0:
            # Skip if head partitioning is ambiguous for this tensor shape.
            continue

        head_dim = out_features // int(num_attention_heads)
        per_head_mass = fisher_abs.reshape(num_attention_heads, head_dim, in_features).sum(dim=(1, 2))

        for head_index, mass_value in enumerate(per_head_mass.tolist()):
            rows.append(
                {
                    'layer_index': layer_index,
                    'layer_label': format_layer_label(layer_index),
                    'projection': projection_name,
                    'head_index': int(head_index),
                    'mass': float(mass_value),
                }
            )

    head_projection_df = pd.DataFrame(rows)
    if head_projection_df.empty:
        return head_projection_df

    # Aggregate across q/k/v so each row becomes a single head in one layer.
    head_df = (
        head_projection_df
        .groupby(['layer_index', 'layer_label', 'head_index'], as_index=False)
        .agg(mass=('mass', 'sum'))
    )

    attention_total_mass = float(head_df['mass'].sum())
    head_df['global_mass_share'] = 0.0 if total_mass <= 0.0 else (head_df['mass'] / total_mass)
    head_df['attention_mass_share'] = (
        0.0 if attention_total_mass <= 0.0 else (head_df['mass'] / attention_total_mass)
    )

    head_df = head_df.sort_values('global_mass_share', ascending=False).reset_index(drop=True)
    return head_df


TOPK_LAYER_ROWS = 20
TOPK_LAYER_MODULE_ROWS = 30
TOPK_HEAD_ROWS = 40

# Use task model config as attention-head reference.
reference_model_path = Path(TASK_SPECS[0].model_path)
num_attention_heads = resolve_num_attention_heads(reference_model_path=reference_model_path)
print(f"Attention head count from config: {num_attention_heads}")

for task_spec in TASK_SPECS:
    task_name = task_spec.name
    fisher_path = Path(FISHER_OUTPUT_PATHS[task_name])

    print(f"\n=== Fisher dominant-parameter diagnostics: task={task_name} ===")

    fisher_tensors = load_fisher_dictionary_for_task(task_name=task_name, fisher_path=fisher_path)
    parameter_df, total_mass = collect_parameter_mass_dataframe(fisher_tensors=fisher_tensors)

    if parameter_df.empty:
        print(f"No Fisher tensors found for task={task_name}; skipping.")
        continue

    layer_df, layer_module_df = build_layer_module_summaries(parameter_df=parameter_df, total_mass=total_mass)

    print(f"Total Fisher mass: {total_mass:.6e}")
    print(f"Num parameter tensors: {len(parameter_df)}")

    print("\nTop layers by global Fisher mass share:")
    display(layer_df.head(TOPK_LAYER_ROWS)[['layer_label', 'mass_share_global', 'mass', 'numel', 'mean']])

    print("\nTop (layer, module_group) by global Fisher mass share:")
    display(
        layer_module_df.head(TOPK_LAYER_MODULE_ROWS)[
            ['layer_label', 'module_group', 'mass_share_global', 'mass', 'numel', 'mean']
        ]
    )

    print("\nTop parameter tensors by global Fisher mass share:")
    display(
        parameter_df
        .sort_values('mass_share_global', ascending=False)
        .head(20)[['parameter_name', 'layer_label', 'module_group', 'mass_share_global', 'mean', 'max']]
        .reset_index(drop=True)
    )

    head_df = pd.DataFrame()
    if num_attention_heads is not None and num_attention_heads > 0:
        head_df = collect_attention_head_mass(
            fisher_tensors=fisher_tensors,
            num_attention_heads=num_attention_heads,
            total_mass=total_mass,
        )

    if not head_df.empty:
        print("\nTop attention heads by global Fisher mass share (aggregated over q/k/v):")
        display(
            head_df.head(TOPK_HEAD_ROWS)[
                ['layer_label', 'head_index', 'global_mass_share', 'attention_mass_share', 'mass']
            ]
        )
    else:
        print("\nAttention head analysis unavailable (config missing or no q/k/v matrices matched).")

    # Visualization panel: top layers + layer-module heatmap + head heatmap.
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))

    # Panel 1: top layers bar chart.
    top_layer_plot_df = layer_df.head(TOPK_LAYER_ROWS).sort_values('mass_share_global', ascending=True)
    axes[0].barh(top_layer_plot_df['layer_label'], top_layer_plot_df['mass_share_global'])
    axes[0].set_title(f"Top layers by Fisher mass share ({task_name})")
    axes[0].set_xlabel('Global Fisher mass share')
    axes[0].set_ylabel('Layer')

    # Panel 2: layer x module-group heatmap.
    layer_module_pivot = (
        layer_module_df
        .pivot(index='layer_label', columns='module_group', values='mass_share_global')
        .fillna(0.0)
    )

    # Sort layer labels numerically for readability, keeping non-transformer at the end.
    sorted_layer_labels = sorted(
        layer_module_pivot.index.tolist(),
        key=lambda label: (9999 if label == 'non_transformer' else int(label[1:])),
    )
    layer_module_pivot = layer_module_pivot.loc[sorted_layer_labels]

    heatmap_module = axes[1].imshow(layer_module_pivot.to_numpy(), aspect='auto', cmap='magma')
    axes[1].set_title(f"Layer x module Fisher mass share ({task_name})")
    axes[1].set_xlabel('Module group')
    axes[1].set_ylabel('Layer')
    axes[1].set_xticks(range(len(layer_module_pivot.columns)))
    axes[1].set_xticklabels(layer_module_pivot.columns.tolist(), rotation=45, ha='right')
    axes[1].set_yticks(range(len(layer_module_pivot.index)))
    axes[1].set_yticklabels(layer_module_pivot.index.tolist())
    fig.colorbar(heatmap_module, ax=axes[1], fraction=0.046, pad=0.04)

    # Panel 3: layer x head heatmap for q/k/v projections.
    if not head_df.empty:
        head_pivot = head_df.pivot(index='layer_label', columns='head_index', values='global_mass_share').fillna(0.0)
        sorted_head_layers = sorted(
            head_pivot.index.tolist(),
            key=lambda label: (9999 if label == 'non_transformer' else int(label[1:])),
        )
        head_pivot = head_pivot.loc[sorted_head_layers]

        heatmap_head = axes[2].imshow(head_pivot.to_numpy(), aspect='auto', cmap='viridis')
        axes[2].set_title(f"Attention head Fisher mass share ({task_name})")
        axes[2].set_xlabel('Head index')
        axes[2].set_ylabel('Layer')
        axes[2].set_xticks(range(len(head_pivot.columns)))
        axes[2].set_xticklabels(head_pivot.columns.tolist())
        axes[2].set_yticks(range(len(head_pivot.index)))
        axes[2].set_yticklabels(head_pivot.index.tolist())
        fig.colorbar(heatmap_head, ax=axes[2], fraction=0.046, pad=0.04)
    else:
        axes[2].axis('off')
        axes[2].text(
            0.5,
            0.5,
            'Head-level analysis unavailable',
            ha='center',
            va='center',
            fontsize=12,
        )

    plt.tight_layout()
    plt.show()


In [ ]:
# Sequential stage-2 Fisher activation overlap analysis
# Goal:
#   Verify whether Fisher hotspots in sequential stage-2 IF training (math -> if)
#   are activated at similar locations as single-task IF Fisher hotspots.
#
# Key comparisons:
#   1) Fisher overlap: `if_single` vs `if_seq_stage2`.
#   2) Context overlap: with `math` Fisher as reference.
#   3) Optional delta alignment: `|theta_seq_if - theta_math|` vs stage-2 Fisher.

from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Sequence, Tuple

import matplotlib.pyplot as plt


SEQUENTIAL_IF_MODEL_PATH = Path(
    '/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math-if/global_step_50/actor/huggingface'
)
SEQUENTIAL_IF_TASK_NAME = 'if_seq_stage2'
SEQUENTIAL_IF_FISHER_PATH = RUNTIME.output_root / 'fisher_diagonal' / f'fisher_diag_{SEQUENTIAL_IF_TASK_NAME}.pt'
SEQUENTIAL_IF_FISHER_SUMMARY_PATH = (
    RUNTIME.output_root / 'fisher_diagonal' / f'fisher_diag_{SEQUENTIAL_IF_TASK_NAME}_summary.json'
)

# Safety toggles:
# - Keep False to avoid accidental long distributed jobs in notebook execution.
COMPUTE_SEQUENTIAL_FISHER_IF_MISSING = False
# - Optional heavy analysis that loads two full models on CPU for delta comparison.
RUN_STAGE2_DELTA_ALIGNMENT = False

TOP_FRACTIONS: Tuple[float, ...] = (0.01, 0.05, 0.10)
PAIRWISE_COMPARISONS: Tuple[Tuple[str, str], ...] = (
    ('if_single', 'if_seq_stage2'),
    ('math', 'if_seq_stage2'),
    ('math', 'if_single'),
)


def ensure_sequential_stage2_fisher(
    model_path: Path,
    fisher_output_path: Path,
    fisher_summary_path: Path,
    compute_if_missing: bool,
) -> Path:
    """Ensure Fisher artifact exists for sequential stage-2 IF model.

    Args:
        model_path: HF checkpoint path of sequential IF model (trained from math init).
        fisher_output_path: Destination path for sequential Fisher `.pt` dictionary.
        fisher_summary_path: Destination path for distributed worker summary `.json`.
        compute_if_missing: If True, run distributed Fisher job when file is missing.

    Returns:
        Existing or newly generated Fisher artifact path.

    Raises:
        FileNotFoundError: If artifact is missing and compute is disabled.
    """

    if fisher_output_path.exists():
        print(f"Sequential Fisher already exists: {fisher_output_path}")
        return fisher_output_path

    if not compute_if_missing:
        raise FileNotFoundError(
            'Sequential stage-2 Fisher file is missing. '
            f'Set COMPUTE_SEQUENTIAL_FISHER_IF_MISSING=True to create it: {fisher_output_path}'
        )

    sequential_task_spec = TaskSpec(
        name=SEQUENTIAL_IF_TASK_NAME,
        model_path=model_path,
        validation_parquet=IF_VALIDATION_PATH,
    )

    print('Running distributed Fisher for sequential stage-2 IF model...')
    run_distributed_fisher_for_task(
        task_spec=sequential_task_spec,
        runtime=RUNTIME,
        fisher_cfg=FISHER_CFG,
        normalized_validation_path=NORMALIZED_VALIDATION_PATHS['if'],
        fisher_output_path=fisher_output_path,
        summary_output_path=fisher_summary_path,
    )
    print(f"Saved sequential Fisher: {fisher_output_path}")

    return fisher_output_path


def build_mass_share_series(
    dataframe: pd.DataFrame,
    key_columns: Sequence[str],
    value_column: str = 'mass_share_global',
) -> pd.Series:
    """Convert grouped table into a normalized mass-share series.

    Why this exists:
        Overlap metrics (cosine/jaccard/correlation) require aligned vectors,
        so this helper canonicalizes each table into a comparable indexed series.

    Args:
        dataframe: Input DataFrame containing key columns and a mass-share column.
        key_columns: Columns used as index key (e.g., `['layer_label']`).
        value_column: Numeric column representing global mass share.

    Returns:
        `pd.Series` indexed by key tuples or scalar keys, normalized to sum 1 when possible.
    """

    if dataframe.empty:
        return pd.Series(dtype=np.float64)

    grouped = dataframe.groupby(list(key_columns), as_index=False).agg(value=(value_column, 'sum'))

    if len(key_columns) == 1:
        index_values = grouped[key_columns[0]].tolist()
    else:
        index_values = [tuple(row) for row in grouped[list(key_columns)].itertuples(index=False, name=None)]

    series = pd.Series(grouped['value'].to_numpy(dtype=np.float64), index=index_values, dtype=np.float64)

    total = float(series.sum())
    if total > 0.0:
        series = series / total

    return series.sort_values(ascending=False)


def align_mass_share_series(series_a: pd.Series, series_b: pd.Series) -> Tuple[np.ndarray, np.ndarray, pd.Index]:
    """Align two indexed mass-share series on union index.

    Args:
        series_a: Left mass-share series.
        series_b: Right mass-share series.

    Returns:
        Tuple of aligned arrays `(a_values, b_values, aligned_index)`.
    """

    aligned_index = series_a.index.union(series_b.index)
    a_values = series_a.reindex(aligned_index, fill_value=0.0).to_numpy(dtype=np.float64)
    b_values = series_b.reindex(aligned_index, fill_value=0.0).to_numpy(dtype=np.float64)
    return a_values, b_values, aligned_index


def compute_cosine_similarity(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    """Compute cosine similarity with robust zero-vector handling."""

    denominator = float(np.linalg.norm(vector_a) * np.linalg.norm(vector_b))
    if denominator <= 0.0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / denominator)


def compute_top_fraction_jaccard(series_a: pd.Series, series_b: pd.Series, top_fraction: float) -> float:
    """Compute Jaccard overlap of top-fraction activated keys.

    Args:
        series_a: Left mass-share series.
        series_b: Right mass-share series.
        top_fraction: Fraction in (0, 1], e.g., 0.05 for top 5% keys.

    Returns:
        Jaccard index between top-key sets.
    """

    if series_a.empty and series_b.empty:
        return 1.0
    if series_a.empty or series_b.empty:
        return 0.0

    k_a = int(max(1, np.ceil(len(series_a) * float(top_fraction))))
    k_b = int(max(1, np.ceil(len(series_b) * float(top_fraction))))

    set_a = set(series_a.head(k_a).index.tolist())
    set_b = set(series_b.head(k_b).index.tolist())

    union_size = len(set_a.union(set_b))
    if union_size == 0:
        return 1.0
    return float(len(set_a.intersection(set_b)) / union_size)


def compute_pair_metrics(
    left_name: str,
    right_name: str,
    left_series: pd.Series,
    right_series: pd.Series,
    granularity: str,
    top_fractions: Iterable[float],
) -> Dict[str, Any]:
    """Compute overlap/correlation metrics for one pair at one granularity.

    Args:
        left_name: Left source name.
        right_name: Right source name.
        left_series: Left mass-share series.
        right_series: Right mass-share series.
        granularity: Granularity label (`parameter`, `layer`, `layer_module`, `head`).
        top_fractions: Fractions used for top-k Jaccard overlap.

    Returns:
        Metrics dictionary ready for DataFrame construction.
    """

    left_values, right_values, aligned_index = align_mass_share_series(left_series, right_series)

    left_pd = pd.Series(left_values)
    right_pd = pd.Series(right_values)

    pearson_corr = float(left_pd.corr(right_pd, method='pearson')) if len(aligned_index) > 1 else np.nan
    spearman_corr = float(left_pd.corr(right_pd, method='spearman')) if len(aligned_index) > 1 else np.nan

    result: Dict[str, Any] = {
        'left': left_name,
        'right': right_name,
        'granularity': granularity,
        'num_keys_union': int(len(aligned_index)),
        'cosine_similarity': compute_cosine_similarity(left_values, right_values),
        'pearson': pearson_corr,
        'spearman': spearman_corr,
    }

    for fraction in top_fractions:
        key = f"top_{fraction * 100:.1f}pct_jaccard".replace('.', '_')
        result[key] = compute_top_fraction_jaccard(
            series_a=left_series,
            series_b=right_series,
            top_fraction=fraction,
        )

    return result


def collect_stage2_delta_mass_dataframe(
    math_model_path: Path,
    sequential_if_model_path: Path,
    torch_dtype: torch.dtype,
) -> Tuple[pd.DataFrame, float]:
    """Compute absolute stage-2 delta magnitude table: |theta_seq_if - theta_math|.

    Why this exists:
        The user requested stage-2 delta isolation via `sequential_model - math_model`.
        This table lets us compare where stage-2 weights moved versus where stage-2
        Fisher is concentrated.

    Args:
        math_model_path: Stage-1 math model checkpoint path.
        sequential_if_model_path: Stage-2 sequential IF model checkpoint path.
        torch_dtype: dtype for model loading.

    Returns:
        Tuple of:
        - Per-parameter delta-mass DataFrame with same schema as Fisher table.
        - Total absolute delta mass.
    """

    math_model, _ = load_causal_lm(model_name_or_path=math_model_path, torch_dtype=torch_dtype, device='cpu')
    seq_model, _ = load_causal_lm(
        model_name_or_path=sequential_if_model_path,
        torch_dtype=torch_dtype,
        device='cpu',
    )

    validate_parameter_compatibility({'math': math_model, 'if_seq_stage2': seq_model})

    math_named = dict(math_model.named_parameters())

    rows = []
    total_mass = 0.0

    with torch.no_grad():
        for parameter_name, seq_parameter in seq_model.named_parameters():
            if not torch.is_floating_point(seq_parameter.data):
                continue

            math_parameter = math_named[parameter_name]
            delta_abs = (
                seq_parameter.data.detach().to(torch.float32)
                - math_parameter.data.detach().to(torch.float32)
            ).abs()

            parameter_mass = float(delta_abs.sum().item())
            parameter_numel = int(delta_abs.numel())
            layer_index = extract_layer_index(parameter_name)
            normalized_layer_index = int(layer_index) if layer_index is not None else -1

            rows.append(
                {
                    'parameter_name': parameter_name,
                    'layer_index': normalized_layer_index,
                    'layer_label': format_layer_label(normalized_layer_index),
                    'module_group': classify_module_group(parameter_name),
                    'numel': parameter_numel,
                    'mass': parameter_mass,
                    'mean': float(parameter_mass / max(parameter_numel, 1)),
                    'max': float(delta_abs.max().item()) if parameter_numel > 0 else 0.0,
                }
            )

            total_mass += parameter_mass

    # Free model memory explicitly because this optional path is heavy.
    del math_model
    del seq_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    delta_df = pd.DataFrame(rows)
    if total_mass > 0.0 and not delta_df.empty:
        delta_df['mass_share_global'] = delta_df['mass'] / total_mass
    else:
        delta_df['mass_share_global'] = 0.0

    return delta_df, total_mass


# -----------------------------------------------------------------------------
# Step 1: Ensure sequential stage-2 Fisher exists (optionally compute if missing).
# -----------------------------------------------------------------------------
sequential_fisher_path = ensure_sequential_stage2_fisher(
    model_path=SEQUENTIAL_IF_MODEL_PATH,
    fisher_output_path=SEQUENTIAL_IF_FISHER_PATH,
    fisher_summary_path=SEQUENTIAL_IF_FISHER_SUMMARY_PATH,
    compute_if_missing=COMPUTE_SEQUENTIAL_FISHER_IF_MISSING,
)

# -----------------------------------------------------------------------------
# Step 2: Build comparable Fisher summaries across math / if_single / if_seq_stage2.
# -----------------------------------------------------------------------------
fisher_sources: Dict[str, Path] = {
    'math': Path(FISHER_OUTPUT_PATHS['math']),
    'if_single': Path(FISHER_OUTPUT_PATHS['if']),
    'if_seq_stage2': sequential_fisher_path,
}

reference_model_path = Path(TASK_SPECS[0].model_path)
num_attention_heads = resolve_num_attention_heads(reference_model_path=reference_model_path)

analysis_by_source: Dict[str, Dict[str, Any]] = {}

for source_name, fisher_path in fisher_sources.items():
    fisher_tensors = load_fisher_dictionary_for_task(task_name=source_name, fisher_path=fisher_path)
    parameter_df, total_mass = collect_parameter_mass_dataframe(fisher_tensors=fisher_tensors)
    layer_df, layer_module_df = build_layer_module_summaries(parameter_df=parameter_df, total_mass=total_mass)

    head_df = pd.DataFrame()
    if num_attention_heads is not None and num_attention_heads > 0:
        head_df = collect_attention_head_mass(
            fisher_tensors=fisher_tensors,
            num_attention_heads=num_attention_heads,
            total_mass=total_mass,
        )

    analysis_by_source[source_name] = {
        'fisher_path': fisher_path,
        'total_mass': total_mass,
        'parameter_df': parameter_df,
        'layer_df': layer_df,
        'layer_module_df': layer_module_df,
        'head_df': head_df,
    }

# -----------------------------------------------------------------------------
# Step 3: Compute overlap metrics to test “same activation locations?” directly.
# -----------------------------------------------------------------------------
metric_rows = []

for left_name, right_name in PAIRWISE_COMPARISONS:
    left_data = analysis_by_source[left_name]
    right_data = analysis_by_source[right_name]

    series_parameter_left = build_mass_share_series(left_data['parameter_df'], key_columns=['parameter_name'])
    series_parameter_right = build_mass_share_series(right_data['parameter_df'], key_columns=['parameter_name'])

    series_layer_left = build_mass_share_series(left_data['layer_df'], key_columns=['layer_label'])
    series_layer_right = build_mass_share_series(right_data['layer_df'], key_columns=['layer_label'])

    series_layer_module_left = build_mass_share_series(
        left_data['layer_module_df'], key_columns=['layer_label', 'module_group']
    )
    series_layer_module_right = build_mass_share_series(
        right_data['layer_module_df'], key_columns=['layer_label', 'module_group']
    )

    metric_rows.append(
        compute_pair_metrics(
            left_name=left_name,
            right_name=right_name,
            left_series=series_parameter_left,
            right_series=series_parameter_right,
            granularity='parameter_tensor',
            top_fractions=TOP_FRACTIONS,
        )
    )
    metric_rows.append(
        compute_pair_metrics(
            left_name=left_name,
            right_name=right_name,
            left_series=series_layer_left,
            right_series=series_layer_right,
            granularity='layer',
            top_fractions=TOP_FRACTIONS,
        )
    )
    metric_rows.append(
        compute_pair_metrics(
            left_name=left_name,
            right_name=right_name,
            left_series=series_layer_module_left,
            right_series=series_layer_module_right,
            granularity='layer_module',
            top_fractions=TOP_FRACTIONS,
        )
    )

    left_head_df = left_data['head_df']
    right_head_df = right_data['head_df']
    if not left_head_df.empty and not right_head_df.empty:
        series_head_left = build_mass_share_series(left_head_df, key_columns=['layer_label', 'head_index'])
        series_head_right = build_mass_share_series(right_head_df, key_columns=['layer_label', 'head_index'])

        metric_rows.append(
            compute_pair_metrics(
                left_name=left_name,
                right_name=right_name,
                left_series=series_head_left,
                right_series=series_head_right,
                granularity='attention_head',
                top_fractions=TOP_FRACTIONS,
            )
        )

metrics_df = pd.DataFrame(metric_rows)

print('Sequential stage-2 overlap metrics (higher is more similar activation positions):')
display(metrics_df.sort_values(['left', 'right', 'granularity']).reset_index(drop=True))

# Highlight the primary question explicitly: if_single vs if_seq_stage2.
focus_df = metrics_df[
    (metrics_df['left'] == 'if_single')
    & (metrics_df['right'] == 'if_seq_stage2')
].sort_values('granularity').reset_index(drop=True)

print('\nFocus pair: if_single vs if_seq_stage2')
display(focus_df)

# -----------------------------------------------------------------------------
# Step 4: Visual diagnostics for primary pair (if_single vs if_seq_stage2).
# -----------------------------------------------------------------------------
if_single_layers = build_mass_share_series(
    analysis_by_source['if_single']['layer_df'],
    key_columns=['layer_label'],
)
if_seq_layers = build_mass_share_series(
    analysis_by_source['if_seq_stage2']['layer_df'],
    key_columns=['layer_label'],
)

aligned_layer_index = if_single_layers.index.union(if_seq_layers.index)
layer_plot_df = pd.DataFrame(
    {
        'if_single': if_single_layers.reindex(aligned_layer_index, fill_value=0.0),
        'if_seq_stage2': if_seq_layers.reindex(aligned_layer_index, fill_value=0.0),
    }
)

# Keep human-friendly numeric layer order with non-transformer at the end.
def _layer_sort_key(label: str) -> Tuple[int, int]:
    if str(label) == 'non_transformer':
        return (1, 9999)
    return (0, int(str(label)[1:]))

layer_plot_df = layer_plot_df.loc[sorted(layer_plot_df.index.tolist(), key=_layer_sort_key)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(layer_plot_df.index.tolist(), layer_plot_df['if_single'].to_numpy(), label='if_single', marker='o')
axes[0].plot(
    layer_plot_df.index.tolist(),
    layer_plot_df['if_seq_stage2'].to_numpy(),
    label='if_seq_stage2',
    marker='o',
)
axes[0].set_title('Layer Fisher mass-share profile')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Global Fisher mass share')
axes[0].tick_params(axis='x', rotation=60)
axes[0].legend()

axes[1].scatter(
    layer_plot_df['if_single'].to_numpy(),
    layer_plot_df['if_seq_stage2'].to_numpy(),
    alpha=0.85,
)
max_axis = float(max(layer_plot_df['if_single'].max(), layer_plot_df['if_seq_stage2'].max(), 1e-12))
axes[1].plot([0.0, max_axis], [0.0, max_axis], 'k--', linewidth=1.0)
axes[1].set_title('Layer share scatter: if_single vs if_seq_stage2')
axes[1].set_xlabel('if_single layer share')
axes[1].set_ylabel('if_seq_stage2 layer share')

plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------------
# Step 5 (optional): stage-2 delta magnitude alignment (seq_if - math) vs Fisher.
# -----------------------------------------------------------------------------
if RUN_STAGE2_DELTA_ALIGNMENT:
    print('\nRunning optional stage-2 delta alignment: |theta_seq_if - theta_math| vs stage-2 Fisher')

    delta_df, delta_total_mass = collect_stage2_delta_mass_dataframe(
        math_model_path=MATH_MODEL_PATH,
        sequential_if_model_path=SEQUENTIAL_IF_MODEL_PATH,
        torch_dtype=resolve_torch_dtype(RUNTIME.model_dtype_name),
    )
    delta_layer_df, delta_layer_module_df = build_layer_module_summaries(
        parameter_df=delta_df,
        total_mass=delta_total_mass,
    )

    seq_layer_series = build_mass_share_series(
        analysis_by_source['if_seq_stage2']['layer_df'],
        key_columns=['layer_label'],
    )
    delta_layer_series = build_mass_share_series(delta_layer_df, key_columns=['layer_label'])

    seq_vals, delta_vals, aligned_idx = align_mass_share_series(seq_layer_series, delta_layer_series)

    delta_alignment_summary = pd.DataFrame(
        [
            {
                'comparison': 'stage2_fisher_vs_abs_weight_delta',
                'granularity': 'layer',
                'num_keys_union': int(len(aligned_idx)),
                'cosine_similarity': compute_cosine_similarity(seq_vals, delta_vals),
                'pearson': float(pd.Series(seq_vals).corr(pd.Series(delta_vals), method='pearson'))
                if len(aligned_idx) > 1
                else np.nan,
                'spearman': float(pd.Series(seq_vals).corr(pd.Series(delta_vals), method='spearman'))
                if len(aligned_idx) > 1
                else np.nan,
            }
        ]
    )

    print('Stage-2 Fisher vs |seq_if - math| delta alignment summary:')
    display(delta_alignment_summary)

    print('Top layers by stage-2 delta mass share:')
    display(delta_layer_df.sort_values('mass_share_global', ascending=False).head(20))

    print('Top (layer, module_group) by stage-2 delta mass share:')
    display(delta_layer_module_df.sort_values('mass_share_global', ascending=False).head(30))
else:
    print('\nRUN_STAGE2_DELTA_ALIGNMENT=False, skipped |seq_if - math| delta alignment.')


## Output Artifacts

After execution, artifacts are saved under:

- `.../Qwen3-1.7B-fisher-merge/fisher_diagonal/` for per-task Fisher tensors and summaries
- `.../Qwen3-1.7B-fisher-merge/fisher_merge_if_math/` for merged checkpoint
- `.../Qwen3-1.7B-fisher-merge/metadata/` for run manifests and merge summary


In [ ]:
# IF Fisher top-k sparsification checkpoints for base->IF update
#
# Goal:
#   Save three sparse IF checkpoints where only the globally top Fisher coordinates
#   (0.1%, 1%, 5%) are updated from base -> IF task vector.
#
# Update rule (element-wise):
#   theta_sparse = theta_base + 1[F_if >= threshold] * (theta_if - theta_base)
#
# Why this implementation:
#   - We avoid flattening all Fisher tensors at once because full concatenation is
#     memory-heavy for 1B+ parameter models.
#   - We estimate the global threshold from a uniform sample, then apply the mask
#     on full tensors to produce practical sparse checkpoints for downstream eval.

from typing import Any, Dict, Mapping, Tuple

from datetime import datetime
from pathlib import Path


def load_if_fisher_dictionary_for_sparse_merge(fisher_path: Path) -> Mapping[str, torch.Tensor]:
    """Load IF Fisher diagonal tensors used for sparsification.

    Args:
        fisher_path: Path to IF Fisher `.pt` dictionary.

    Returns:
        Mapping from parameter name to Fisher tensor on CPU.

    Raises:
        FileNotFoundError: If the Fisher artifact is missing.
        TypeError: If deserialized object is not a mapping.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f"IF Fisher file was not found: {fisher_path}")

    fisher_obj = torch.load(fisher_path, map_location='cpu')
    if not isinstance(fisher_obj, Mapping):
        raise TypeError(f"IF Fisher file must contain a mapping, got {type(fisher_obj)}")

    return fisher_obj


def count_total_fisher_elements(fisher_tensors: Mapping[str, torch.Tensor]) -> int:
    """Count total scalar Fisher elements across all parameter tensors.

    Args:
        fisher_tensors: Parameter-name keyed Fisher tensor mapping.

    Returns:
        Total scalar element count.
    """

    total_numel = 0
    for fisher_tensor in fisher_tensors.values():
        total_numel += int(fisher_tensor.numel())
    return int(total_numel)


def sample_global_fisher_values(
    fisher_tensors: Mapping[str, torch.Tensor],
    total_numel: int,
    sample_size: int,
    seed: int,
) -> np.ndarray:
    """Sample Fisher values uniformly over global parameter index space.

    Args:
        fisher_tensors: Parameter-name keyed Fisher tensor mapping.
        total_numel: Total scalar Fisher element count.
        sample_size: Number of scalar values to sample.
        seed: RNG seed for reproducibility.

    Returns:
        1D float32 NumPy array of sampled Fisher magnitudes.

    Notes:
        - Sampling is with replacement to keep memory usage stable.
        - This avoids materializing one giant flattened tensor for the full model.
    """

    if total_numel <= 0 or sample_size <= 0:
        return np.zeros(0, dtype=np.float32)

    effective_sample_size = int(min(sample_size, total_numel))
    rng = np.random.default_rng(seed=seed)

    # Draw sorted global indices so we can stream over tensors once.
    sampled_global_indices = np.sort(
        rng.integers(low=0, high=total_numel, size=effective_sample_size, dtype=np.int64)
    )

    sampled_values = np.empty(effective_sample_size, dtype=np.float32)
    write_cursor = 0
    tensor_offset = 0

    for fisher_tensor in fisher_tensors.values():
        fisher_flat = fisher_tensor.detach().to(torch.float32).abs().reshape(-1)
        tensor_numel = int(fisher_flat.numel())

        # Find index segment that belongs to the current tensor slice.
        left = np.searchsorted(sampled_global_indices, tensor_offset, side='left')
        right = np.searchsorted(sampled_global_indices, tensor_offset + tensor_numel, side='left')

        if right > left:
            local_indices = sampled_global_indices[left:right] - tensor_offset
            local_index_tensor = torch.from_numpy(local_indices.astype(np.int64))
            local_values = fisher_flat.index_select(dim=0, index=local_index_tensor)

            next_cursor = write_cursor + (right - left)
            sampled_values[write_cursor:next_cursor] = local_values.cpu().numpy()
            write_cursor = next_cursor

        tensor_offset += tensor_numel

    if write_cursor != effective_sample_size:
        raise RuntimeError(
            f"Sampling bookkeeping mismatch: expected {effective_sample_size}, got {write_cursor}"
        )

    return sampled_values


def estimate_global_fisher_threshold(
    fisher_tensors: Mapping[str, torch.Tensor],
    keep_ratio: float,
    sample_size: int,
    seed: int,
) -> Tuple[float, int, int]:
    """Estimate global Fisher threshold for top-ratio sparsification.

    Args:
        fisher_tensors: Parameter-name keyed Fisher tensor mapping.
        keep_ratio: Fraction of scalar coordinates to keep (e.g., 0.01 for top 1%).
        sample_size: Number of sampled values used for threshold estimation.
        seed: RNG seed for sampling reproducibility.

    Returns:
        Tuple `(threshold, total_numel, sampled_count)`.

    Raises:
        ValueError: If keep_ratio is outside (0, 1].
    """

    if keep_ratio <= 0.0 or keep_ratio > 1.0:
        raise ValueError(f"keep_ratio must be in (0, 1], got {keep_ratio}")

    total_numel = count_total_fisher_elements(fisher_tensors)
    sampled_values = sample_global_fisher_values(
        fisher_tensors=fisher_tensors,
        total_numel=total_numel,
        sample_size=sample_size,
        seed=seed,
    )

    if sampled_values.size == 0:
        return 0.0, int(total_numel), int(sampled_values.size)

    # Top keep_ratio => threshold at (1 - keep_ratio) quantile.
    threshold = float(np.quantile(sampled_values, q=(1.0 - float(keep_ratio))))
    return threshold, int(total_numel), int(sampled_values.size)


def apply_sparse_if_update_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    fisher_tensors: Mapping[str, torch.Tensor],
    fisher_threshold: float,
) -> Dict[str, Any]:
    """Apply IF sparse delta update to base model in-place.

    Args:
        base_model: Base model to be updated and saved.
        if_model: IF fine-tuned model that provides `theta_if`.
        fisher_tensors: IF Fisher diagonal mapping keyed by parameter name.
        fisher_threshold: Global threshold; keep coordinates where Fisher >= threshold.

    Returns:
        Summary dictionary with kept/total element counts and realized keep ratio.

    Raises:
        ValueError: If parameter names or tensor shapes are incompatible.
    """

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())

    if set(base_named.keys()) != set(if_named.keys()):
        raise ValueError('Base and IF parameter keys are incompatible for sparse update.')

    kept_elements = 0
    total_elements = 0

    with torch.no_grad():
        for param_name, base_param in base_named.items():
            if not torch.is_floating_point(base_param.data):
                continue

            if param_name not in fisher_tensors:
                raise ValueError(f"Missing IF Fisher tensor for parameter: {param_name}")

            if_param = if_named[param_name]
            fisher_abs = fisher_tensors[param_name].detach().to(torch.float32).abs()

            if tuple(fisher_abs.shape) != tuple(base_param.shape):
                raise ValueError(
                    f"Fisher shape mismatch for '{param_name}': "
                    f"fisher={tuple(fisher_abs.shape)} vs model={tuple(base_param.shape)}"
                )

            # Compute IF task-vector delta from the same base anchor.
            base_fp32 = base_param.data.detach().to(torch.float32)
            if_fp32 = if_param.data.detach().to(torch.float32)
            delta_if = if_fp32 - base_fp32

            # Keep only high-Fisher coordinates, revert others to base.
            sparse_mask = fisher_abs >= float(fisher_threshold)
            sparse_update = delta_if * sparse_mask.to(torch.float32)
            merged_tensor = base_fp32 + sparse_update
            base_param.data.copy_(merged_tensor.to(base_param.dtype))

            kept_elements += int(sparse_mask.sum().item())
            total_elements += int(sparse_mask.numel())

    return {
        'fisher_threshold': float(fisher_threshold),
        'kept_elements': int(kept_elements),
        'total_elements': int(total_elements),
        'realized_keep_ratio': float(kept_elements / max(total_elements, 1)),
    }


def keep_ratio_to_tag(keep_ratio: float) -> str:
    """Format keep ratio into a filesystem-safe tag.

    Args:
        keep_ratio: Scalar ratio in (0, 1].

    Returns:
        Tag string such as `top_0p1pct`.
    """

    pct_value = keep_ratio * 100.0
    pct_str = f"{pct_value:.3f}".rstrip('0').rstrip('.')
    return f"top_{pct_str.replace('.', 'p')}pct"


def save_sparse_if_checkpoint(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save sparse IF checkpoint and metadata.

    Args:
        model: Updated sparse model.
        tokenizer: Tokenizer saved next to model.
        output_dir: Output directory for checkpoint artifacts.
        metadata: JSON-serializable metadata dictionary.

    Returns:
        None.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)

    metadata_path = output_dir / 'merge_metadata.json'
    if 'save_json' in globals():
        # Reuse notebook utility so Path/dtype conversion stays consistent.
        save_json(dict(metadata), metadata_path)
    else:
        with metadata_path.open('w', encoding='utf-8') as file:
            json.dump(dict(metadata), file, indent=2, ensure_ascii=False)


SPARSE_KEEP_RATIOS: Tuple[float, ...] = (0.001, 0.01, 0.05)
SPARSE_THRESHOLD_SAMPLE_SIZE = 2_000_000
SPARSE_OUTPUT_ROOT = RUNTIME.output_root / 'if_sparse_topk_fisher'

if_fisher_path = Path(FISHER_OUTPUT_PATHS['if'])
if_fisher_tensors = load_if_fisher_dictionary_for_sparse_merge(fisher_path=if_fisher_path)

merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)
if_model_for_delta, _ = load_causal_lm(
    model_name_or_path=IF_MODEL_PATH,
    torch_dtype=merge_dtype,
    device='cpu',
)

sparse_run_rows = []

for keep_ratio in SPARSE_KEEP_RATIOS:
    threshold, total_numel, sampled_count = estimate_global_fisher_threshold(
        fisher_tensors=if_fisher_tensors,
        keep_ratio=keep_ratio,
        sample_size=SPARSE_THRESHOLD_SAMPLE_SIZE,
        seed=RUNTIME.seed,
    )

    sparse_model, sparse_tokenizer = load_causal_lm(
        model_name_or_path=RUNTIME.base_model_id,
        torch_dtype=merge_dtype,
        device='cpu',
    )

    sparse_summary = apply_sparse_if_update_inplace(
        base_model=sparse_model,
        if_model=if_model_for_delta,
        fisher_tensors=if_fisher_tensors,
        fisher_threshold=threshold,
    )

    output_tag = keep_ratio_to_tag(keep_ratio)
    output_dir = SPARSE_OUTPUT_ROOT / f'if_delta_sparse_{output_tag}'

    metadata = {
        'created_at': now_iso() if 'now_iso' in globals() else datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'method': 'if_delta_sparse_by_if_fisher_topk',
        'formula': 'theta_sparse = theta_base + 1[F_if >= threshold] * (theta_if - theta_base)',
        'base_model_id': str(RUNTIME.base_model_id),
        'if_model_path': str(IF_MODEL_PATH),
        'if_fisher_path': str(if_fisher_path),
        'keep_ratio': float(keep_ratio),
        'sample_size_for_threshold': int(SPARSE_THRESHOLD_SAMPLE_SIZE),
        'threshold_estimation_total_numel': int(total_numel),
        'threshold_estimation_sampled_count': int(sampled_count),
        'sparse_summary': sparse_summary,
    }

    save_sparse_if_checkpoint(
        model=sparse_model,
        tokenizer=sparse_tokenizer,
        output_dir=output_dir,
        metadata=metadata,
    )

    sparse_run_rows.append(
        {
            'keep_ratio': float(keep_ratio),
            'fisher_threshold': float(sparse_summary['fisher_threshold']),
            'realized_keep_ratio': float(sparse_summary['realized_keep_ratio']),
            'kept_elements': int(sparse_summary['kept_elements']),
            'total_elements': int(sparse_summary['total_elements']),
            'output_dir': str(output_dir),
        }
    )

    print(
        f"Saved sparse IF checkpoint | keep_ratio={keep_ratio:.4f} "
        f"| realized={sparse_summary['realized_keep_ratio']:.6f} "
        f"| threshold={sparse_summary['fisher_threshold']:.6e} "
        f"| path={output_dir}"
    )

    del sparse_model
    del sparse_tokenizer
    gc.collect()

summary_payload = {
    'created_at': now_iso() if 'now_iso' in globals() else datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'method': 'if_delta_sparse_by_if_fisher_topk',
    'if_fisher_path': str(if_fisher_path),
    'keep_ratios': [float(x) for x in SPARSE_KEEP_RATIOS],
    'sample_size_for_threshold': int(SPARSE_THRESHOLD_SAMPLE_SIZE),
    'runs': sparse_run_rows,
}
summary_path = RUNTIME.output_root / 'metadata' / 'if_sparse_topk_fisher_summary.json'

if 'save_json' in globals():
    save_json(summary_payload, summary_path)
else:
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    with summary_path.open('w', encoding='utf-8') as file:
        json.dump(summary_payload, file, indent=2, ensure_ascii=False)

display(pd.DataFrame(sparse_run_rows))
print(f"Saved sparse IF run summary: {summary_path}")


del if_model_for_delta
del if_fisher_tensors
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


